# Netz Stroke-Transformer HMER (HMER-Baseline-002)

**Policy: HARD LANDING.** Real online handwritten math only (Google **MathWriting** or **CROHME**). No synthetic data, no fallback generators. The MathWriting `synthetic/` and `symbols/` folders are **excluded** on purpose.

### What changed vs. 001
- **Real loader:** reads **InkML** (the format MathWriting / CROHME actually ship in) as well as `.jsonl`; prefers the `normalizedLabel` annotation; uses the official `train / valid / test` folders when present.
- **ExpRate fixed:** compared as **token lists** (was space-joined string vs raw LaTeX, which almost never matched). Also reports token error rate and ExpRate with <=1 / <=2 errors.
- **Token budget 160** (was 32) and stroke budget 512; samples that don't fit are dropped and counted.
- **Vocabulary from the training split only.**
- Bigger model (d=256, 4 enc + 3 dec layers, tied embeddings), proper **padding masks**, label smoothing, warmup+cosine LR, AMP, augmentation, **validation loss + periodic validation ExpRate**, best-checkpoint saving, early stopping.
- **ONNX verified**: logits match PyTorch for several shapes **and** full greedy decoding through ONNX Runtime matches PyTorch token-for-token.

### How to run on Kaggle
1. **File -> Import Notebook** -> `train_hmer.ipynb`
2. **Nothing to attach:** with Settings -> **Internet: ON** the official Google MathWriting (2.9 GB, human-written train/valid/test only) is downloaded automatically. You can still attach MathWriting/CROHME InkML yourself and it will be used instead.
3. Accelerator: **GPU T4 x2 / P100**. **Run All**.
4. Download `netz_hmer.onnx`, `netz_hmer_vocab.json`, `netz_hmer_meta.json`, `netz_hmer_feature_test.json` into `Netz/public/models/`.

Default is a 80k-sample training subset to keep run time reasonable; set `NETZ_MAX_TRAIN=0` for everything.
Env overrides: `NETZ_MATH_DIR`, `NETZ_EPOCHS`, `NETZ_MAX_TRAIN`, `NETZ_MAX_EVAL`.

**Inference note:** the ONNX model is a single forward pass (strokes + tokens so far -> logits). The web app must run the autoregressive loop itself (start with `<sos>`=1, append argmax, stop at `<eos>`=2 or 160 tokens).

In [ ]:
# 1. Environment & configuration
import os, sys, re, math, json, time, random, hashlib, copy, subprocess
import xml.etree.ElementTree as ET
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def _pip(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
try:
    import onnx, onnxruntime as ort
except ImportError:
    _pip(['onnx', 'onnxruntime'])
    import onnx, onnxruntime as ort

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

MAX_TGT = 160                                   # max tokens incl. <sos>/<eos>
CFG = dict(
    max_len=512,                                # max stroke points per sample (subsampled above this)
    epochs=int(os.environ.get('NETZ_EPOCHS', 25)),
    batch_size=int(os.environ.get('NETZ_BATCH', 32)),
    lr=5e-4, warmup_steps=1000, patience=6,
    max_train=(int(os.environ.get('NETZ_MAX_TRAIN', 80000)) or None),
    max_eval=(int(os.environ.get('NETZ_MAX_EVAL', 2000)) or None),
    num_workers=int(os.environ.get('NETZ_WORKERS', 2)),
    scale_mode='max',
)
print(f'HMER environment: {device} | PyTorch {torch.__version__} | AMP={use_amp}')
print('Config:', CFG)

# ======================================================================
# Netz canonical stroke features  --  IDENTICAL in all three notebooks.
# Mirror this exactly in the web app (JS) and verify with the exported
# *_feature_test.json file.
#
# Raw ink array: float32 (N, 4) = [x, y, t, stroke_id]
# Feature columns (8): 0 x | 1 y | 2 dx | 3 dy | 4 dt | 5 stroke_end | 6 stroke_start | 7 speed
# ======================================================================
FEATURE_DIM = 8
FEATURE_SPEC = ['x', 'y', 'dx', 'dy', 'dt', 'stroke_end', 'stroke_start', 'speed']


def strokes_to_array(strokes):
    """strokes: list of strokes; a point is dict(x,y[,t]) or a sequence (x, y[, t])."""
    rows = []
    for sid, stroke in enumerate(strokes):
        for pt in stroke:
            if isinstance(pt, dict):
                x, y, t = pt['x'], pt['y'], pt.get('t', None)
            else:
                x, y = pt[0], pt[1]
                t = pt[2] if len(pt) >= 3 else None
            rows.append((float(x), float(y), np.nan if t is None else float(t), float(sid)))
    arr = np.asarray(rows, dtype=np.float64).reshape(-1, 4)
    if len(arr) and np.isfinite(arr[:, 2]).any():
        arr[:, 2] -= np.nanmin(arr[:, 2])          # relative time keeps float32 precise
    return arr.astype(np.float32)


def _to_seconds(t):
    """Auto-detect ms vs s from the median sampling gap; fall back to 10 ms/sample if no timing."""
    t = np.asarray(t, dtype=np.float64)
    n = len(t)
    fallback = np.arange(n, dtype=np.float64) * 0.01
    if n < 2 or not np.isfinite(t).all():
        return fallback
    d = np.diff(t)
    pos = d[d > 0]
    if len(pos) == 0:
        return fallback
    return t / 1000.0 if np.median(pos) > 1.0 else t


def extract_features(arr, max_len, scale_mode='max'):
    """Return float32 (n<=max_len, 8). No padding here (batching pads and tracks real lengths).
    scale_mode: 'max'    -> divide by max(width, height)      (math, router)
                'height' -> divide by max(height, 0.03*width) (text lines)"""
    arr = np.asarray(arr)
    n = len(arr)
    if n == 0:
        return np.zeros((1, FEATURE_DIM), dtype=np.float32)
    t_sec = _to_seconds(arr[:, 2])
    if n > max_len:                                   # deterministic, JS-reproducible subsampling
        idx = np.floor(np.arange(max_len) * (n - 1) / (max_len - 1) + 0.5).astype(np.int64)
        arr, t_sec = arr[idx], t_sec[idx]
        n = max_len
    x = arr[:, 0].astype(np.float64)
    y = arr[:, 1].astype(np.float64)
    sid = arr[:, 3]
    x0, y0 = x.min(), y.min()
    w, h = x.max() - x0, y.max() - y0
    scale = max(h, 0.03 * w, 1e-6) if scale_mode == 'height' else max(w, h, 1e-6)
    xn, yn = (x - x0) / scale, (y - y0) / scale
    dx = np.diff(xn, prepend=xn[0])
    dy = np.diff(yn, prepend=yn[0])
    dt = np.diff(t_sec, prepend=t_sec[0])
    dt = np.where(np.arange(n) == 0, 0.0, np.maximum(dt, 1e-3))
    speed = np.sqrt(dx * dx + dy * dy) / np.maximum(dt, 1e-3)
    start = np.ones(n)
    start[1:] = (sid[1:] != sid[:-1])
    end = np.ones(n)
    end[:-1] = (sid[:-1] != sid[1:])
    feats = np.stack([xn, yn, dx, dy, np.minimum(dt * 10.0, 1.0), end, start, np.minimum(speed / 5.0, 1.0)], axis=1)
    return feats.astype(np.float32)


# ---------------- augmentation on raw ink (train only) ----------------
def thin_points(arr, rng, keep_min=0.7):
    """Randomly drop points (simulates other sampling rates); stroke endpoints are always kept."""
    n = len(arr)
    if n < 8:
        return arr
    sid = arr[:, 3]
    keep = rng.random(n) < rng.uniform(keep_min, 1.0)
    keep[0] = keep[-1] = True
    keep[1:] |= (sid[1:] != sid[:-1])
    keep[:-1] |= (sid[:-1] != sid[1:])
    return arr[keep]


def augment_array(arr, rng, thin=True):
    a = np.array(arr, dtype=np.float64)
    if len(a) == 0:
        return arr
    if thin:
        a = thin_points(a, rng)
    x, y = a[:, 0], a[:, 1]
    cx, cy = x.mean(), y.mean()
    ang = rng.uniform(-0.08, 0.08)
    sx, sy = rng.uniform(0.85, 1.15), rng.uniform(0.85, 1.15)
    shear = rng.uniform(-0.25, 0.25)
    xr, yr = x - cx, y - cy
    x2 = xr * np.cos(ang) - yr * np.sin(ang)
    y2 = xr * np.sin(ang) + yr * np.cos(ang)
    x2 = x2 + shear * y2
    a[:, 0] = x2 * sx + cx
    a[:, 1] = y2 * sy + cy
    span = max(np.ptp(a[:, 0]), np.ptp(a[:, 1]), 1e-6)
    a[:, :2] += rng.normal(0.0, 0.002 * span, size=(len(a), 2))
    return a.astype(np.float32)


# ---------------- shared metrics ----------------
def edit_distance(a, b):
    """Levenshtein distance between two sequences (strings or lists)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


# ---------------- parity test vector for the JS port ----------------
TEST_STROKES = [
    # case 1: tiny, ms timestamps, pen-up gap between the two strokes
    [[[0, 0, 0], [10, 5, 10], [20, 8, 20], [30, 20, 30]],
     [[5, 40, 100], [15, 42, 110], [25, 50, 120]]],
    # case 2: 3000 points -> exercises the subsampling path
    [[[i, round(100 * math.sin(i / 50), 3), i * 10] for i in range(1500)],
     [[i, round(100 * math.cos(i / 50), 3) + 300, 20000 + i * 10] for i in range(1500)]],
]


def write_feature_test(path, max_len, scale_mode):
    cases = []
    for strokes in TEST_STROKES:
        f = extract_features(strokes_to_array(strokes), max_len, scale_mode)
        cases.append({'strokes': strokes, 'features': np.round(f, 6).tolist()})
    with open(path, 'w') as fh:
        json.dump({'feature_spec': FEATURE_SPEC, 'max_len': max_len, 'scale_mode': scale_mode,
                   'tolerance': 1e-4, 'cases': cases}, fh)
    print(f'Wrote {path}  (feed these strokes through the JS extractor; max abs diff must be < 1e-4)')

# ---------------- dataset discovery helpers (archives + diagnostics) ----------------
import tarfile, zipfile
from collections import Counter

def list_inputs(max_dirs=40):
    """Print what is actually attached under /kaggle/input so a failed search is easy to debug."""
    base = '/kaggle/input'
    if not os.path.isdir(base):
        print('No /kaggle/input here (not running on Kaggle?). Set NETZ_IAM_DIR / NETZ_MATH_DIR to your data folders.')
        return
    print('Attached inputs:', sorted(os.listdir(base)) or 'NONE - click "+ Add Input" in the right sidebar')
    shown = 0
    for dp, dn, fn in os.walk(base):
        if dp[len(base):].count(os.sep) > 4:
            dn[:] = []
            continue
        exts = Counter(os.path.splitext(f)[1].lower() or '(none)' for f in fn)
        print(f'  {dp}   subdirs={len(dn)} files={len(fn)} {dict(exts.most_common(4))}')
        shown += 1
        if shown >= max_dirs:
            print('  ...'); break

def extract_archives(src_roots, dest=None):
    """Extract only the ink/transcript files from .zip/.tar/.tar.gz/.tgz found in src_roots (skips 'synthetic')."""
    wanted = ('.inkml', '.xml', '.txt', '.jsonl', '.mlf')
    if dest is None:
        dest = '/kaggle/working/netz_extracted' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/.cache/netz/extracted')
    out = []
    for root in src_roots:
        for dp, _, fn in os.walk(root):
            for f in fn:
                low = f.lower()
                if not low.endswith(('.tar.gz', '.tgz', '.tar', '.zip')):
                    continue
                target = os.path.join(dest, re.sub(r'\W+', '_', f))
                full = os.path.join(dp, f)
                if not os.path.isdir(target):
                    os.makedirs(target, exist_ok=True)
                    print(f'Extracting {full} -> {target} (ink/transcript files only)')
                    try:
                        if low.endswith('.zip'):
                            with zipfile.ZipFile(full) as zf:
                                names = [n for n in zf.namelist() if n.lower().endswith(wanted) and 'synthetic' not in n.lower()]
                                zf.extractall(target, members=names)
                        else:
                            with tarfile.open(full) as tf:
                                members = [m for m in tf if m.isfile() and m.name.lower().endswith(wanted) and 'synthetic' not in m.name.lower()]
                                try:
                                    tf.extractall(target, members=members, filter='data')
                                except TypeError:
                                    tf.extractall(target, members=members)
                    except Exception as e:
                        print('  could not extract:', e)
                out.append(target)
    return out


MATHWRITING_URL = os.environ.get('NETZ_MW_URL', 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz')   # official, 2.9 GB

def download_mathwriting(dest='/kaggle/working/mathwriting_real'):
    """Official Google MathWriting (needs Internet: ON). Streams the .tgz and keeps ONLY human-written train/valid/test InkML
    (synthetic and symbols are skipped, which also saves ~GBs of disk). Returns the folder or None."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/mathwriting_real')
    if os.path.isdir(dest) and any(f.endswith('.inkml') for _, _, fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    tgz = os.path.join(dest, 'mathwriting.tgz')
    try:
        print(f'Downloading MathWriting from {MATHWRITING_URL} (2.9 GB, takes a few minutes) ...')
        urllib.request.urlretrieve(MATHWRITING_URL, tgz, _progress_hook())
        n = 0
        with tarfile.open(tgz, 'r|gz') as tf:                       # streaming: no seeking, low memory
            for m in tf:
                parts = [p.lower() for p in m.name.split('/')]
                if m.isfile() and m.name.lower().endswith('.inkml') and any(p in ('train', 'valid', 'test') for p in parts[:-1]) \
                        and not any(p in ('synthetic', 'symbols') for p in parts):
                    tf.extract(m, dest, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
                    n += 1
        os.remove(tgz)
        print(f'Extracted {n} human-written InkML files to {dest}')
        return dest if n else None
    except Exception as e:
        print(f'MathWriting download failed: {e}\n  -> turn Internet ON in Settings, or attach MathWriting as an input.')
        return None


def _progress_hook():
    state = {'next': 0}
    def hook(blocks, bsize, total):
        if total > 0:
            pct = min(100, int(blocks * bsize * 100 / total))
            if pct >= state['next']:
                print(f'   download {pct}%', flush=True)
                state['next'] = pct + 10
    return hook

IAM_KAGGLE_HANDLE = os.environ.get('NETZ_IAM_KAGGLE', '')     # e.g. 'yourname/iam-ondb' = YOUR private upload of IAM-OnDB
IAM_URLS = [u for u in os.environ.get('NETZ_IAM_URLS', '').split(',') if u.strip()]   # direct links to lineStrokes-all.tar.gz, ascii-all.tar.gz, ...

def download_iam(dest='/kaggle/working/iam_downloaded'):
    """IAM-OnDB is licence-gated (free registration at fki.tic.heia-fr.ch), so there is NO public auto-download.
    This fetches it from a place YOU control: a private Kaggle dataset (NETZ_IAM_KAGGLE) or direct URLs (NETZ_IAM_URLS)."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/iam_downloaded')
    os.makedirs(dest, exist_ok=True)
    got = []
    if IAM_KAGGLE_HANDLE:
        try:
            import kagglehub
            print(f'Downloading your IAM-OnDB upload from Kaggle: {IAM_KAGGLE_HANDLE}')
            got.append(kagglehub.dataset_download(IAM_KAGGLE_HANDLE))
        except Exception as e:
            print(f'  kagglehub download failed: {e}')
    for url in IAM_URLS:
        try:
            fn = os.path.join(dest, os.path.basename(url.split('?')[0]) or 'iam_file')
            print(f'Downloading {url}')
            urllib.request.urlretrieve(url.strip(), fn, _progress_hook())
            got.append(dest)
        except Exception as e:
            print(f'  download failed: {e}')
    return sorted(set(got))

IAM_HELP = (
    'IAM-OnDB cannot be auto-downloaded (licence: free registration required).\n'
    '  ONE-TIME SETUP, then it is automatic forever:\n'
    '   1. (The old unibe.ch address is retired.) Register (free) at https://fki.tic.heia-fr.ch/databases/iam-on-line-handwriting-database and download the On-Line DB files:\n'
    '      lineStrokes-all.tar.gz, ascii-all.tar.gz and the 4 split files trainset.txt, testset_v.txt, testset_t.txt, testset_f.txt\n'
    '   2. Kaggle -> Datasets -> New Dataset -> upload those files -> keep it PRIVATE.\n'
    '   3. In this notebook either click "+ Add Input" and attach it, OR set  NETZ_IAM_KAGGLE=yourname/your-dataset-slug  (auto-download).')

In [ ]:
# 2. Locate real math ink (HARD LANDING: NO FALLBACK)
EXCLUDE_DIRS = ('synthetic', 'symbols', 'symbol')

def path_split(path):
    parts = [p.lower() for p in re.split(r'[\\/]', path)]
    if any(p in EXCLUDE_DIRS for p in parts):
        return 'EXCLUDED'
    for p in parts[:-1]:
        if p == 'train': return 'train'
        if p in ('valid', 'val', 'validation'): return 'val'
        if p == 'test': return 'test'
    return None

def find_math_files(extra=()):
    roots = [r for r in [os.environ.get('NETZ_MATH_DIR'), '/kaggle/input', 'data'] if r and os.path.isdir(r)] + list(extra)
    inkml, jsonl = [], []
    for root in roots:
        for dp, _, files in os.walk(root):
            for f in files:
                low = f.lower()
                full = os.path.join(dp, f)
                if low.endswith('.inkml'):
                    inkml.append(full)
                elif low.endswith('.jsonl'):
                    jsonl.append(full)
    return roots, inkml, jsonl

roots, inkml_files, jsonl_files = find_math_files()
if not inkml_files and not jsonl_files:
    roots, inkml_files, jsonl_files = find_math_files(extract_archives(roots))   # maybe the data is zipped
inkml_files = [p for p in inkml_files if path_split(p) != 'EXCLUDED']
if not inkml_files and not jsonl_files:
    mw = download_mathwriting()                                                  # official MathWriting, needs Internet ON
    if mw:
        roots, inkml_files, jsonl_files = find_math_files([mw])
        inkml_files = [p for p in inkml_files if path_split(p) != 'EXCLUDED']
if not inkml_files and not jsonl_files:
    list_inputs()
    listing = []
    for r in roots:
        for dp, dn, fn in os.walk(r):
            listing.append(f'{dp}  ({len(fn)} files)')
            if len(listing) > 25: break
    raise FileNotFoundError(
        '\n' + '=' * 70 + '\n[FATAL - HARD LANDING]\nNo real online math data found (.inkml / .jsonl).\n'
        'Attach MathWriting or CROHME (InkML) or set NETZ_MATH_DIR. Searched: ' + str(roots) +
        '\nFirst folders seen:\n  ' + '\n  '.join(listing) + '\n' + '=' * 70)
print(f'[Real Data Confirmed] {len(inkml_files)} InkML files, {len(jsonl_files)} jsonl files (synthetic/symbols excluded)')

In [ ]:
# 3. InkML / jsonl parsing
NUM_RE = re.compile(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?')

def parse_trace_text(text):
    if "'" in text or '"' in text:             # CROHME differential coding - not supported, skip sample
        return None
    pts = []
    for seg in text.split(','):
        nums = NUM_RE.findall(seg)
        if len(nums) >= 3:
            pts.append((float(nums[0]), float(nums[1]), float(nums[2])))
        elif len(nums) == 2:
            pts.append((float(nums[0]), float(nums[1]), float('nan')))
    return pts

def clean_label(s):
    s = (s or '').strip()
    if s.startswith('$') and s.endswith('$') and len(s) >= 2:
        s = s.strip('$').strip()
    return s

def parse_inkml(path):
    root = ET.parse(path).getroot()
    strokes, norm, raw, bad = [], None, None, False
    for el in root.iter():
        tag = el.tag.split('}')[-1]
        if tag == 'trace':
            pts = parse_trace_text(el.text or '')
            if pts is None:
                bad = True
            elif pts:
                strokes.append(pts)
        elif tag == 'annotation':
            typ = el.get('type', '')
            if typ == 'normalizedLabel':
                norm = el.text
            elif typ in ('label', 'truth'):
                raw = el.text
    label = clean_label(norm or raw)
    return (None, None) if (bad or not strokes or not label) else (strokes, label)

def parse_jsonl_line(line):
    e = json.loads(line)
    ink = e.get('ink', e.get('strokes', []))
    label = clean_label(e.get('label', e.get('truth', '')))
    strokes = [[tuple(p[:3]) if len(p) >= 3 else (p[0], p[1], float('nan')) for p in s if len(p) >= 2] for s in ink]
    strokes = [s for s in strokes if s]
    return (strokes, label) if strokes and label else (None, None)

In [ ]:
# 4. LaTeX tokenizer + vocabulary (built from the TRAIN split only)
TOKEN_RE = re.compile(r'\\[A-Za-z]+|\\.|\d|[A-Za-z]|\S')
SPECIAL = ['<pad>', '<sos>', '<eos>', '<unk>']

def tokenize_latex(s):
    return TOKEN_RE.findall(s or '')

class LatexVocab:
    def __init__(self, tokens):
        self.tokens = SPECIAL + [t for t in tokens if t not in SPECIAL]
        self.t2i = {t: i for i, t in enumerate(self.tokens)}
        self.pad, self.sos, self.eos, self.unk = 0, 1, 2, 3
    def __len__(self):
        return len(self.tokens)
    def encode(self, toks):
        return [self.sos] + [self.t2i.get(t, self.unk) for t in toks] + [self.eos]
    def decode_tokens(self, ids):
        out = []
        for i in ids:
            if i == self.eos: break
            if i not in (self.pad, self.sos):
                out.append(self.tokens[i] if i < len(self.tokens) else '<unk>')
        return out

In [ ]:
# 5. Build splits (official folders if present) and load real ink
def hash_split(key):
    v = int(hashlib.md5(key.encode()).hexdigest(), 16) % 100
    return 'train' if v < 90 else ('val' if v < 95 else 'test')

by_split = {'train': [], 'val': [], 'test': []}
for p in inkml_files:
    s = path_split(p)
    if s:
        by_split[s].append(p)
have_official = len(by_split['train']) > 0 and len(by_split['test']) > 0
if have_official and len(by_split['val']) == 0:                       # carve validation from train
    tr = by_split['train']
    by_split['val'] = [p for p in tr if hash_split(p) == 'val']
    by_split['train'] = [p for p in tr if hash_split(p) != 'val']
if not have_official:
    by_split = {'train': [], 'val': [], 'test': []}
    for p in inkml_files:
        by_split[hash_split(p)].append(p)
    print('No official train/valid/test folders -> hash split by file.')
else:
    print('Using official folder splits.')
print('InkML files per split:', {k: len(v) for k, v in by_split.items()})

rng = random.Random(SEED)
for k in by_split:
    rng.shuffle(by_split[k])
cap = {'train': CFG['max_train'], 'val': CFG['max_eval'], 'test': CFG['max_eval']}

raw_items = {'train': [], 'val': [], 'test': []}
stats = Counter()
t0 = time.time()
for split in ('train', 'val', 'test'):
    for n_seen, p in enumerate(by_split[split]):
        if cap[split] and len(raw_items[split]) >= cap[split]:
            break
        try:
            strokes, label = parse_inkml(p)
        except Exception:
            stats['parse_error'] += 1
            continue
        if strokes is None:
            stats['no_strokes_or_label_or_differential'] += 1
            continue
        toks = tokenize_latex(label)
        if len(toks) + 2 > MAX_TGT:
            stats['too_many_tokens'] += 1
            continue
        arr = strokes_to_array(strokes)
        if len(arr) < 2:
            stats['too_short'] += 1
            continue
        raw_items[split].append((arr, toks))
        if (n_seen + 1) % 10000 == 0:
            print(f'  {split}: parsed {n_seen + 1} ({time.time() - t0:.0f}s)')

# optional jsonl sources (original format) - only used when InkML provided nothing at all
if all(len(v) == 0 for v in raw_items.values()):
    for jp in jsonl_files:
        with open(jp, 'r', encoding='utf-8') as fh:
            for line in fh:
                if not line.strip():
                    continue
                try:
                    strokes, label = parse_jsonl_line(line)
                except Exception:
                    continue
                if strokes is None:
                    continue
                toks = tokenize_latex(label)
                if len(toks) + 2 > MAX_TGT:
                    continue
                split = hash_split(label + str(len(strokes)))
                if cap[split] and len(raw_items[split]) >= cap[split]:
                    continue
                raw_items[split].append((strokes_to_array(strokes), toks))
        if all(len(v) > 0 for v in raw_items.values()):
            break

print({k: len(v) for k, v in raw_items.items()}, dict(stats), f'| load time {time.time() - t0:.0f}s')
for k in ('train', 'val', 'test'):
    if len(raw_items[k]) == 0:
        raise RuntimeError(f'Split "{k}" is empty. Parsing failed - inspect a sample .inkml and adapt parse_inkml().')

counter = Counter(t for _, toks in raw_items['train'] for t in toks)
vocab = LatexVocab([t for t, _ in counter.most_common()])
unk_rate = sum(1 for _, toks in raw_items['test'] for t in toks if t not in vocab.t2i) / max(1, sum(len(toks) for _, toks in raw_items['test']))
print(f'Vocab (train only): {len(vocab)} tokens | test tokens unseen in train: {unk_rate * 100:.3f}%')

class MathInk(Dataset):
    def __init__(self, data, train):
        self.data, self.train, self.rng = data, train, None
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        arr, toks = self.data[i]
        if self.train:
            if self.rng is None:
                self.rng = np.random.default_rng()
            arr = augment_array(arr, self.rng, thin=True)
        feats = extract_features(arr, CFG['max_len'], CFG['scale_mode'])
        return torch.from_numpy(feats), torch.tensor(vocab.encode(toks), dtype=torch.long), toks

def collate(batch):
    feats, ids, toks = zip(*batch)
    B = len(feats)
    lengths = torch.tensor([len(f) for f in feats])
    T = int(lengths.max())
    x = torch.zeros(B, T, FEATURE_DIM)
    for i, f in enumerate(feats):
        x[i, :len(f)] = f
    src_pad = torch.arange(T)[None, :] >= lengths[:, None]               # True = padding
    L = max(len(t) for t in ids)
    y = torch.full((B, L), vocab.pad, dtype=torch.long)
    for i, t in enumerate(ids):
        y[i, :len(t)] = t
    return x, src_pad, y, list(toks)

def make_loader(split, shuffle):
    return DataLoader(MathInk(raw_items[split], train=(split == 'train')), batch_size=CFG['batch_size'], shuffle=shuffle,
                      collate_fn=collate, num_workers=CFG['num_workers'], drop_last=(split == 'train'))

train_loader, val_loader, test_loader = make_loader('train', True), make_loader('val', False), make_loader('test', False)

In [ ]:
# 6. Sanity check on real samples (mandatory)
print('--- [Sanity check] ---')
chk = MathInk(raw_items['train'], train=False)
pts, ntok = [], []
for i in range(min(200, len(chk))):
    f, ids, toks = chk[i]
    assert f.ndim == 2 and f.shape[1] == FEATURE_DIM and torch.isfinite(f).all()
    assert vocab.decode_tokens(ids.tolist()) == [t if t in vocab.t2i else '<unk>' for t in toks]
    pts.append(len(f)); ntok.append(len(toks))
print(f'points/expr: mean={np.mean(pts):.0f} p95={np.percentile(pts, 95):.0f} | tokens/expr: mean={np.mean(ntok):.1f} max={max(ntok)}')
print('sample 0:', ' '.join(chk[0][2]))
print('--- [Sanity check passed] ---')

In [ ]:
# 7. Stroke-Transformer model (pre-norm encoder over strokes + autoregressive LaTeX decoder, ONNX-safe attention)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MHA(nn.Module):
    """Plain multi-head attention built from basic ops so the ONNX export keeps dynamic sequence lengths
    (torch's nn.MultiheadAttention / nn.TransformerDecoder bake the dummy length into the exported graph)."""
    def __init__(self, d, heads, dropout):
        super().__init__()
        self.h, self.dh, self.p = heads, d // heads, dropout
        self.q, self.k, self.v, self.o = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
    def forward(self, x, mem, blocked=None):             # blocked: bool, broadcastable to (B,H,Tq,Tk); True = not allowed
        B = x.size(0)
        q = self.q(x).view(B, -1, self.h, self.dh).transpose(1, 2)
        k = self.k(mem).view(B, -1, self.h, self.dh).transpose(1, 2)
        v = self.v(mem).view(B, -1, self.h, self.dh).transpose(1, 2)
        att = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dh)
        if blocked is not None:
            att = att.masked_fill(blocked, float('-inf'))
        att = F.dropout(att.softmax(-1), self.p, self.training)
        return self.o(torch.matmul(att, v).transpose(1, 2).reshape(B, -1, self.h * self.dh))

class FFN(nn.Module):
    def __init__(self, d, ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff, d))
    def forward(self, x):
        return self.net(x)

class EncLayer(nn.Module):
    def __init__(self, d, heads, ff, dropout):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.att, self.ffn, self.drop = MHA(d, heads, dropout), FFN(d, ff, dropout), nn.Dropout(dropout)
    def forward(self, x, blocked):
        h = self.n1(x)
        x = x + self.drop(self.att(h, h, blocked))
        return x + self.drop(self.ffn(self.n2(x)))

class DecLayer(nn.Module):
    def __init__(self, d, heads, ff, dropout):
        super().__init__()
        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.self_att, self.cross_att = MHA(d, heads, dropout), MHA(d, heads, dropout)
        self.ffn, self.drop = FFN(d, ff, dropout), nn.Dropout(dropout)
    def forward(self, x, memory, causal, mem_blocked):
        h = self.n1(x)
        x = x + self.drop(self.self_att(h, h, causal))
        x = x + self.drop(self.cross_att(self.n2(x), memory, mem_blocked))
        return x + self.drop(self.ffn(self.n3(x)))

class StrokeTransformerHMER(nn.Module):
    def __init__(self, vocab_size, in_features=FEATURE_DIM, d_model=256, nhead=8, enc_layers=4, dec_layers=3,
                 ff=1024, dropout=0.1, max_src=1024, max_tgt=MAX_TGT + 8):
        super().__init__()
        self.d_model = d_model
        self.in_proj = nn.Sequential(nn.Linear(in_features, d_model), nn.LayerNorm(d_model), nn.GELU())
        self.local = nn.Sequential(nn.Conv1d(d_model, d_model, 5, padding=2), nn.GELU())      # local stroke-shape mixing
        self.pos_src, self.pos_tgt = PositionalEncoding(d_model, max_src), PositionalEncoding(d_model, max_tgt)
        self.enc_layers = nn.ModuleList([EncLayer(d_model, nhead, ff, dropout) for _ in range(enc_layers)])
        self.enc_norm = nn.LayerNorm(d_model)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.token_emb.weight, 0.0, d_model ** -0.5)
        self.dec_layers = nn.ModuleList([DecLayer(d_model, nhead, ff, dropout) for _ in range(dec_layers)])
        self.dec_norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.fc_out.weight = self.token_emb.weight                                              # tied embeddings
        self.register_buffer('causal', torch.triu(torch.ones(max_tgt, max_tgt, dtype=torch.bool), diagonal=1), persistent=False)

    def encode(self, strokes, src_pad=None):
        h = self.in_proj(strokes)
        h = self.pos_src(h + self.local(h.transpose(1, 2)).transpose(1, 2))
        blocked = None if src_pad is None else src_pad[:, None, None, :]
        for layer in self.enc_layers:
            h = layer(h, blocked)
        return self.enc_norm(h)

    def decode(self, tgt, memory, src_pad=None):
        sz = tgt.size(1)
        x = self.pos_tgt(self.token_emb(tgt) * math.sqrt(self.d_model))
        causal = self.causal[:sz, :sz][None, None]
        mem_blocked = None if src_pad is None else src_pad[:, None, None, :]
        for layer in self.dec_layers:
            x = layer(x, memory, causal, mem_blocked)
        return self.fc_out(self.dec_norm(x))

    def forward(self, strokes, tgt, src_pad=None):
        return self.decode(tgt, self.encode(strokes, src_pad), src_pad)

    @torch.no_grad()
    def generate(self, strokes, src_pad=None, max_len=MAX_TGT - 1, sos=1, eos=2):
        memory = self.encode(strokes, src_pad)
        cur = torch.full((strokes.size(0), 1), sos, dtype=torch.long, device=strokes.device)
        done = torch.zeros(strokes.size(0), dtype=torch.bool, device=strokes.device)
        for _ in range(max_len):
            nxt = self.decode(cur, memory, src_pad)[:, -1].argmax(-1, keepdim=True)
            cur = torch.cat([cur, nxt], dim=1)
            done |= (nxt.squeeze(1) == eos)
            if done.all():
                break
        return cur

model = StrokeTransformerHMER(len(vocab)).to(device)
print(f'HMER model: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters | vocab {len(vocab)}')

In [ ]:
# 8. Training (teacher forcing, label smoothing, warmup+cosine, AMP, best checkpoint on val loss)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad, label_smoothing=0.1)
plain_ce = nn.CrossEntropyLoss(ignore_index=vocab.pad)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-2, betas=(0.9, 0.98))
total_steps = CFG['epochs'] * max(1, len(train_loader))
def lr_lambda(step):
    if step < CFG['warmup_steps']:
        return (step + 1) / CFG['warmup_steps']
    prog = (step - CFG['warmup_steps']) / max(1, total_steps - CFG['warmup_steps'])
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def decode_batch(x, src_pad, max_len=MAX_TGT - 1):
    model.eval()
    gen = model.generate(x.to(device), src_pad.to(device), max_len=max_len, sos=vocab.sos, eos=vocab.eos)
    return [vocab.decode_tokens(g) for g in gen.cpu().tolist()]

def val_loss():
    model.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for x, src_pad, y, _ in val_loader:
            x, src_pad, y = x.to(device), src_pad.to(device), y.to(device)
            logits = model(x, y[:, :-1], src_pad)
            tot += plain_ce(logits.reshape(-1, len(vocab)).float(), y[:, 1:].reshape(-1)).item(); n += 1
    return tot / max(1, n)

def exprate(loader, max_batches=None):
    preds, refs = [], []
    for b, (x, src_pad, _, toks) in enumerate(loader):
        if max_batches and b >= max_batches: break
        preds.extend(decode_batch(x, src_pad)); refs.extend(toks)
    return preds, refs

def score(preds, refs):
    n = max(1, len(refs))
    errs = [edit_distance(p, r) for p, r in zip(preds, refs)]
    return dict(exprate=sum(e == 0 for e in errs) / n, le1=sum(e <= 1 for e in errs) / n, le2=sum(e <= 2 for e in errs) / n,
                token_error_rate=sum(errs) / max(1, sum(len(r) for r in refs)))

best_vl, best_state, bad = float('inf'), None, 0
for epoch in range(1, CFG['epochs'] + 1):
    model.train(); tot, nb, t0 = 0.0, 0, time.time()
    for x, src_pad, y, _ in train_loader:
        x, src_pad, y = x.to(device), src_pad.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(x, y[:, :-1], src_pad)
        loss = criterion(logits.reshape(-1, len(vocab)).float(), y[:, 1:].reshape(-1))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tot += loss.item(); nb += 1
    vl = val_loss(); flag = ''
    if vl < best_vl:
        best_vl, best_state, bad, flag = vl, copy.deepcopy(model.state_dict()), 0, '  <- best'
    else:
        bad += 1
    msg = f'Epoch {epoch:02d}/{CFG["epochs"]} | train {tot / max(1, nb):.3f} | val CE {vl:.3f} | {time.time() - t0:.0f}s{flag}'
    if epoch % 3 == 0 or epoch == CFG['epochs']:
        p, r = exprate(val_loader, max_batches=16)
        s = score(p, r)
        msg += f' | val ExpRate(subset) {s["exprate"] * 100:.1f}% TER {s["token_error_rate"] * 100:.1f}%'
    print(msg)
    if bad >= CFG['patience']:
        print('Early stopping.'); break
model.load_state_dict(best_state)

In [ ]:
# 9. Untouched test-set evaluation (token-list comparison)
preds, refs = exprate(test_loader)
s = score(preds, refs)
print('=== Untouched Test Set ===')
print(f'ExpRate: {s["exprate"] * 100:.2f}% | <=1 error: {s["le1"] * 100:.2f}% | <=2 errors: {s["le2"] * 100:.2f}% | '
      f'token error rate: {s["token_error_rate"] * 100:.2f}% | {len(refs)} expressions')
for r, p in list(zip(refs, preds))[:5]:
    print('  REF :', ' '.join(r)); print('  PRED:', ' '.join(p)); print()
if s['exprate'] < 0.30:
    print('WARNING: ExpRate < 30% - usable only as a rough baseline. Use the full dataset and more epochs.')

In [ ]:
# 10. Production ONNX export, verified (logits + full autoregressive decode)
model.eval().cpu()

class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, strokes, tgt_tokens):
        return self.m(strokes, tgt_tokens)

wrapper = ExportWrapper(model).eval()
dummy_s, dummy_t = torch.randn(1, 200, FEATURE_DIM), torch.ones(1, 12, dtype=torch.long)
onnx_path = 'netz_hmer.onnx'
kw = dict(export_params=True, opset_version=17, do_constant_folding=True, input_names=['strokes', 'tgt_tokens'], output_names=['logits'],
          dynamic_axes={'strokes': {0: 'batch_size', 1: 'stroke_seq_len'}, 'tgt_tokens': {0: 'batch_size', 1: 'tgt_seq_len'},
                        'logits': {0: 'batch_size', 1: 'tgt_seq_len'}})
try:
    torch.onnx.export(wrapper, (dummy_s, dummy_t), onnx_path, dynamo=False, **kw)
except TypeError:
    torch.onnx.export(wrapper, (dummy_s, dummy_t), onnx_path, **kw)
onnx.checker.check_model(onnx.load(onnx_path))
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

max_diff = 0.0
for T, L in ((60, 3), (250, 20), (CFG['max_len'], 90)):
    xs, ts = torch.randn(1, T, FEATURE_DIM), torch.randint(1, len(vocab), (1, L))
    with torch.no_grad():
        ref = model(xs, ts).numpy()
    out = sess.run(None, {'strokes': xs.numpy(), 'tgt_tokens': ts.numpy()})[0]
    max_diff = max(max_diff, float(np.abs(ref - out).max()))
print(f'ONNX vs PyTorch max |logit diff|: {max_diff:.2e}')
assert max_diff < 1e-2, 'ONNX export does not match PyTorch'

def onnx_greedy(feats, max_len=MAX_TGT - 1):
    cur = np.array([[vocab.sos]], dtype=np.int64)
    for _ in range(max_len):
        lg = sess.run(None, {'strokes': feats, 'tgt_tokens': cur})[0]
        nxt = int(lg[0, -1].argmax())
        cur = np.concatenate([cur, [[nxt]]], axis=1)
        if nxt == vocab.eos: break
    return vocab.decode_tokens(cur[0].tolist())

agree = 0
check_n = min(20, len(raw_items['test']))
for arr, toks in raw_items['test'][:check_n]:
    f = extract_features(arr, CFG['max_len'], CFG['scale_mode'])[None]
    with torch.no_grad():
        pt_tokens = vocab.decode_tokens(model.generate(torch.from_numpy(f), None).tolist()[0])
    agree += int(pt_tokens == onnx_greedy(f))
print(f'Full greedy decode, ONNX == PyTorch on {agree}/{check_n} real test expressions')
assert agree >= 0.9 * check_n, 'ONNX decoding diverges from PyTorch'

with open('netz_hmer_vocab.json', 'w') as fh:
    json.dump(vocab.tokens, fh, indent=2)
with open('netz_hmer_meta.json', 'w') as fh:
    json.dump({'max_len': CFG['max_len'], 'max_tgt': MAX_TGT, 'scale_mode': CFG['scale_mode'], 'pad': 0, 'sos': 1, 'eos': 2, 'unk': 3,
               'feature_spec': FEATURE_SPEC, 'test_exprate': round(s['exprate'], 4), 'test_token_error_rate': round(s['token_error_rate'], 4)}, fh, indent=2)
write_feature_test('netz_hmer_feature_test.json', CFG['max_len'], CFG['scale_mode'])
print(f'Exported {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB), netz_hmer_vocab.json, netz_hmer_meta.json')